# STEP 3. 라벨 생성 + 표본 필터 + 로그 변환

## 라벨 정의
```
순증감(t) = 개업_점포_수(t) − 폐업_점포_수(t)
y(t)      = 1 if 순증감(t+1) < 0 else 0     ← 다음 분기 순감소 전환
```

**주의 두 가지**
- `shift(-1)` 은 반드시 (상권, 업종) 그룹 내에서 **분기 오름차순 정렬 후** 수행
- 분기가 연속하지 않으면 라벨을 만들지 않음

## 필터 근거
점포 5개 미만 칸은 개업·폐업이 거의 없어 라벨이 노이즈에 가깝습니다. 유동인구 결측 행은 H2 매칭이 불가능합니다.

In [2]:
import sys, os
from pathlib import Path

# 노트북이 어디서 열리든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("프로젝트 루트:", ROOT)

프로젝트 루트: c:\Users\spide\ai-data-bootcamp\project\h2_nb


In [3]:
from config import PROC, MIN_STORES

df = pd.read_pickle(PROC / "02_merged.pkl")

def quarter_index(code):
    """20241 → 연속 정수. 분기 연속성 판정용"""
    return (code // 10) * 4 + (code % 10 - 1)

df["분기idx"] = quarter_index(df["기준_년분기_코드"])
df["순증감"] = df["개업_점포_수"] - df["폐업_점포_수"]
df["순감소_현재"] = (df["순증감"] < 0).astype(int)
print(f"입력 {df.shape}")

입력 (160352, 29)


## 3-1. 라벨 생성

In [4]:
df = df.sort_values(["상권_코드", "서비스_업종_코드", "분기idx"]).reset_index(drop=True)
g = df.groupby(["상권_코드", "서비스_업종_코드"], sort=False)

df["y"] = g["순감소_현재"].shift(-1)
df["다음분기idx"] = g["분기idx"].shift(-1)

연속 = (df["다음분기idx"] - df["분기idx"]) == 1
df.loc[~연속, "y"] = np.nan

print(f"라벨 생성 가능 {df['y'].notna().sum():,} / 전체 {len(df):,}")
print(f"(마지막 분기이거나 분기가 끊긴 칸은 라벨 없음)\n")
print(f"무사건 칸(개업=0 & 폐업=0) : {((df['개업_점포_수']==0)&(df['폐업_점포_수']==0)).mean():.1%}")
print(f"점포 {MIN_STORES}개 미만          : {(df['점포_수'] < MIN_STORES).mean():.1%}")

라벨 생성 가능 147,138 / 전체 160,352
(마지막 분기이거나 분기가 끊긴 칸은 라벨 없음)

무사건 칸(개업=0 & 폐업=0) : 66.4%
점포 5개 미만          : 59.4%


## 3-2. 표본 필터 — 단계별 손실 추적

In [5]:
steps = [("원본", df)]
d = df[df["y"].notna()].copy();          steps.append(("라벨 존재", d))
d = d[d["점포_수"] >= MIN_STORES];        steps.append((f"점포 {MIN_STORES}개↑", d))
d = d[d["총_유동인구_수"].notna()];        steps.append(("유동인구 존재", d))
d = d[d["영역_면적"] > 0];                steps.append(("면적 유효", d))

prev = None
for nm, s in steps:
    drop = "" if prev is None else f"  (-{prev-len(s):,})"
    print(f"  {nm:14} {len(s):>8,}{drop}")
    prev = len(s)

d["y"] = d["y"].astype(int)
print(f"\n최종 표본 {len(d):,}행 | 양성률(순감소) {d['y'].mean():.1%}")

  원본              160,352
  라벨 존재           147,138  (-13,214)
  점포 5개↑           59,652  (-87,486)
  유동인구 존재          59,632  (-20)
  면적 유효            59,632  (-0)

최종 표본 59,632행 | 양성률(순감소) 26.6%


## 3-3. 분기별 양성률

In [6]:
display(d.groupby("기준_년분기_코드")["y"].agg(["size", "mean"]).round(3))

,size,mean
기준_년분기_코드,,
20231,4728,0.243
20232,4729,0.279
20233,4713,0.234
20234,4735,0.250
20241,4723,0.271
20242,4721,0.293
20243,4657,0.279
20244,4611,0.310
20251,5534,0.261


## 3-4. 로그 변환

영역_면적이 1,300배 차이나므로 로그 변환은 선택이 아니라 필수입니다.

In [7]:
d["log_점포수"]   = np.log(d["점포_수"])
d["log_유동인구"] = np.log1p(d["총_유동인구_수"])
d["log_면적"]     = np.log(d["영역_면적"])
d["log_집객시설"] = np.log1p(d["집객시설_수"].fillna(0))
d["점포당매출"]   = d["당월_매출_금액"] / d["점포_수"]

display(d[["log_점포수", "log_유동인구", "log_면적", "log_집객시설"]]
        .describe().loc[["mean", "std", "min", "max"]].round(3))

d.to_pickle(PROC / "03_panel.pkl")
print(f"[저장] 03_panel.pkl  {d.shape}")

,log_점포수,log_유동인구,log_면적,log_집객시설
mean,2.570,13.672,11.648,3.189
std,0.839,1.017,0.851,1.016
min,1.609,2.197,7.525,0.000
max,6.516,15.969,14.717,6.389


[저장] 03_panel.pkl  (59632, 36)
